In [1]:
#@title Muat Data Processed & Vectorizer TF-IDF
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

train_df = pd.read_csv("./data/processed/train.csv")
val_df = pd.read_csv("./data/processed/val.csv")

# Deteksi nama kolom teks ('clean_text' atau 'clean')
text_col = "clean_text" if "clean_text" in train_df.columns else "clean"
X_train_text = train_df[text_col].fillna("").astype(str)
X_val_text = val_df[text_col].fillna("").astype(str)
y_train = train_df["label"].astype(int)
y_val = val_df["label"].astype(int)

# Transformasi teks menjadi matriks TF-IDF menggunakan vectorizer yang sudah di-fit di train set
vectorizer = joblib.load("./models/tfidf_vectorizer.joblib")
X_train = vectorizer.transform(X_train_text)
X_val = vectorizer.transform(X_val_text)

print("Jumlah data train :", X_train.shape[0], "| Fitur:", X_train.shape[1])
print("Jumlah data val   :", X_val.shape[0], "| Fitur:", X_val.shape[1])

Jumlah data train : 8446 | Fitur: 5000
Jumlah data val   : 1056 | Fitur: 5000


In [2]:
#title Latih Random Forest dengan parameter awal
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

print("Model Random Forest awal berhasil dilatih.")

Model Random Forest awal berhasil dilatih.


In [3]:
#@title Evaluasi pada validation set
y_val_pred_rf = rf_model.predict(X_val)

rf_metrics = {
    "Model": "Random Forest (Awal)",
    "Accuracy": round(accuracy_score(y_val, y_val_pred_rf), 4),
    "Precision": round(precision_score(y_val, y_val_pred_rf), 4),
    "Recall": round(recall_score(y_val, y_val_pred_rf), 4),
    "F1-Score": round(f1_score(y_val, y_val_pred_rf), 4),
}

for k, v in rf_metrics.items():
    print(f"{k:10s}: {v}")

Model     : Random Forest (Awal)
Accuracy  : 0.9233
Precision : 0.9234
Recall    : 0.8924
F1-Score  : 0.9076


In [4]:
# perbandingan hasil validation set antara Baseline (MultinomialNB) vs Advanced (Random Forest)
nb_model = joblib.load("./models/model_nb_baseline.joblib")
y_val_pred_nb = nb_model.predict(X_val)

nb_metrics = {
    "Model": "Multinomial NB (Baseline)",
    "Accuracy": round(accuracy_score(y_val, y_val_pred_nb), 4),
    "Precision": round(precision_score(y_val, y_val_pred_nb), 4),
    "Recall": round(recall_score(y_val, y_val_pred_nb), 4),
    "F1-Score": round(f1_score(y_val, y_val_pred_nb), 4),
}

comparison_df = pd.DataFrame([nb_metrics, rf_metrics])
display(comparison_df)

,Model,Accuracy,Precision,Recall,F1-Score
0,Multinomial NB (Baseline),0.8693,0.9400,0.7377,0.8266
1,Random Forest (Awal),0.9233,0.9234,0.8924,0.9076
